# Phase 2: Hybrid Retrieval with RRF

This notebook compares the existing FAISS dense retriever, the BM25 lexical retriever, and their explicit Reciprocal Rank Fusion (RRF). No reranking or self-healing logic is used.

## Environment

Replace `REPOSITORY_URL` with your repository URL before running this notebook in Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## Initialize dense, lexical, and hybrid retrieval

Both retrievers use the same controlled Phase 1 corpus. The hybrid retriever requests each ranking and adds `1 / (60 + rank)` for every list in which a document appears.

In [ ]:
from src.rag import (
    BM25Retriever,
    FAISSRetriever,
    HybridRetriever,
    RAGConfig,
    load_documents,
    measure_retrieval_overlap,
)

config = RAGConfig(top_k=3)
documents = load_documents(Path('data/phase1_corpus.json'))
index_dir = Path('data/phase1_faiss_index')

dense_retriever = FAISSRetriever(
    embedding_model_name=config.embedding_model_name,
    device=config.device,
    batch_size=config.embedding_batch_size,
)
if (index_dir / 'documents.faiss').exists():
    dense_retriever.load(index_dir)
    print(f'Loaded FAISS index with {len(dense_retriever.documents)} documents.')
else:
    dense_retriever.build(documents)
    dense_retriever.save(index_dir)
    print(f'Built FAISS index with {len(documents)} documents.')

bm25_retriever = BM25Retriever(documents)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)
print(f'Built BM25 index with {len(bm25_retriever.documents)} documents.')

## Compare rankings and overlap

In [ ]:
def print_results(label, results, score_label):
    print(f'\n{label}')
    for rank, result in enumerate(results, start=1):
        print(f'  Rank {rank}: {result.title} (id={result.id}, {score_label}={result.score:.6f})')
        print(f'    {result.text}')


def compare_retrievers(query: str, top_k: int = 3):
    dense_results = dense_retriever.retrieve(query, top_k=top_k)
    bm25_results = bm25_retriever.retrieve(query, top_k=top_k)
    hybrid_results = hybrid_retriever.retrieve(query, top_k=top_k)
    agreement = measure_retrieval_overlap(dense_results, bm25_results)

    print('\n' + '=' * 100)
    print('Query:', query)
    print_results('Dense results', dense_results, 'cosine')
    print_results('BM25 results', bm25_results, 'BM25')

    print('\nHybrid results')
    for rank, result in enumerate(hybrid_results, start=1):
        print(f'  Rank {rank}: {result.title} (id={result.id}, RRF={result.score:.6f}, dense_rank={result.dense_rank}, bm25_rank={result.bm25_rank})')
        print(f'    {result.text}')

    print('\nDense/BM25 overlap')
    print(f'  Shared IDs: {list(agreement.shared_document_ids)}')
    print(f'  Overlap: {agreement.overlap_count}/{min(len(dense_results), len(bm25_results))} ({agreement.overlap_ratio:.1%})')
    return dense_results, bm25_results, hybrid_results, agreement

In [ ]:
example_queries = [
    # Lexical/exact-match query
    'IPv4 uses 32-bit addresses',
    # Paraphrased query
    'Which observatory circling our planet was named for an astronomer?',
    # A second factual query
    'Where can visitors see Leonardo da Vinci’s famous portrait?',
]

for example_query in example_queries:
    compare_retrievers(example_query, top_k=config.top_k)

## Optional: use hybrid retrieval for generation

`HybridResult` preserves the `id`, `title`, `text`, and `score` interface used by the existing Qwen prompt. To generate with hybrid context, construct `BaselineRAG(hybrid_retriever, LocalQwenGenerator(config), config)`. Qwen is not loaded in this retrieval-focused notebook by default.